## Task 1: Understanding tools

Tool in an AI Agent:


A tool is a function that an AI agent uses to perform tasks such as calculations, web searches, or database queries



Why do agents need tools?


Agents use tools to perform calculations, retrieve information, access external services, and complete tasks more effectively.



Difference between a chatbot and an agent.


A chatbot mainly answers user questions, while an AI agent can select and use tools to perform tasks and achieve goals.

## Task 2: Built-in LangChain tools

In [9]:
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun, ArxivQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper
import wikipedia

# Initialize web search tool
search_tool = DuckDuckGoSearchRun()

result = search_tool.invoke("Latest features of Python")

print(result)

# Initialize Wikipedia tool
wikipedia.wikipedia.USER_AGENT = "my-learning-app/1.0 (your_email@example.com)"

wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

result = wiki_tool.invoke("AI Agents")
print(result)

arxiv_tool = ArxivQueryRun(api_wrapper=ArxivAPIWrapper())

arxiv_result = arxiv_tool.invoke("large language models")

print(arxiv_result)

Python 3.15.0 is the newest major release of the Python programming language, and it contains many new features and optimisations compared to Python 3.14, in 5,643 commits from 1,012 contributors. Some of the major new features and changes in Python 3.14 are: New features PEP 779: Free-threaded Python is officially supported PEP 649: The evaluation of annotations is now deferred, improving the semantics of using annotations. PEP 750: Template string literals (t-strings) for custom string processing, using the familiar syntax of f-strings. The latest Python release notes, changelogs and updates, summarised in plain English - new features, breaking changes, security advisories and deprecations, updated continuously. Python is a high‑level, versatile programming language known for its simplicity, readability, and wide range of applications. It is widely used in web development, data science, artificial intelligence and automation. This article explores the key features and characteristics

## Task 3: Create a custom tool

In [10]:
from langchain_core.tools import tool

@tool
def company_policy_lookup(query: str) -> str:
    """Returns company policy information."""
    # Mock Data
    policies = {
        "leave": "Employees receive 20 paid leaves per year.",
        "work from home": "Employees can work from home 2 days per week.",
        "working hours": "Working hours are 9 AM to 6 PM."
    }
    query = query.lower()
    for key, value in policies.items():
        if key in query:
            return value
    return "No matching company policy found."

print(company_policy_lookup.invoke({"query": "leave"}))
print(company_policy_lookup.invoke({"query": "work from home"}))
print(company_policy_lookup.invoke({"query": "working hours"}))
print(company_policy_lookup.invoke({"query": "salary"}))

Employees receive 20 paid leaves per year.
Employees can work from home 2 days per week.
Working hours are 9 AM to 6 PM.
No matching company policy found.


## Task 4: Create a custom toolkit

In [11]:
from datetime import datetime

# Creating more tools

@tool
def database_query(query: str) -> str:
    """Searches employee database using employee ID."""

    employees = {
        "101": "Manish - Flutter Developer",
        "102": "Rahul - Backend Developer",
        "103": "Priya - UI Designer"
    }

    # Extract the employee ID from the query
    query = query.strip()

    for employee_id, details in employees.items():
        if employee_id in query:
            return details

    return "Employee not found."
    
@tool
def current_datetime() -> str:
    """Returns the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Grouping multiple toolkit
company_toolkit = [
    company_policy_lookup,
    database_query,
    current_datetime
]

print("Custom toolkit created successfully.")

for tool in company_toolkit:
    print(tool.name)

Custom toolkit created successfully.
company_policy_lookup
database_query
current_datetime


In [12]:
# Testing all tools

print(company_policy_lookup.invoke({"query": "leave"}))

print(database_query.invoke({"query": "101"}))

print(current_datetime.invoke({}))

Employees receive 20 paid leaves per year.
Manish - Flutter Developer
2026-10-21 16:39:57


## Task 5: Bind tools to an LLM

In [13]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage

load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-120b")

# Using previous task toolkit which has all three tools
llm_with_tools = llm.bind_tools(company_toolkit)

queries = [
    "employee id 101", 
    "What is the current date and time",
    "How many paid leaves do employees get?"
    ]

for query in queries:
    response = llm_with_tools.invoke([HumanMessage(content=query)])
    print(response.tool_calls)

[{'name': 'database_query', 'args': {'query': 'SELECT * FROM employees WHERE id = 101'}, 'id': 'fc_93041e21-3df1-43ac-bf0c-4c6a1b1db4f4', 'type': 'tool_call'}]
[{'name': 'current_datetime', 'args': {}, 'id': 'fc_cb6737cb-ff90-45fa-91b9-e29004e81e9c', 'type': 'tool_call'}]
[{'name': 'company_policy_lookup', 'args': {'query': 'paid leave entitlement'}, 'id': 'fc_77d78bef-b8e2-49c7-86ce-db3ac035d41b', 'type': 'tool_call'}]


## Task 6: Tool-calling flow

In [14]:
from langchain_core.messages import HumanMessage, ToolMessage

tools_by_name = {tool.name: tool for tool in company_toolkit}

def run_agent(query: str) -> str:
    messages = [HumanMessage(content=query)]

    while True:
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        # Stop when the LLM has no more tool calls
        if not response.tool_calls:
            print("Final Answer:", response.content)
            break

        # Execute each requested tool
        for tool_call in response.tool_calls:
            selected_tool = tools_by_name[tool_call['name']]
            result = selected_tool.invoke(tool_call['args'])

            print('Selected tool: ', tool_call['name'])
            print('Tool Output: ', result)

            messages.append(ToolMessage(content=str(result), tool_call_id=tool_call['id']))

In [15]:
# Test

run_agent("How many paid leaves do employees get?")
run_agent("Find the employee id 101")
run_agent("Tell me the working hours and find the employee id 101")

Selected tool:  company_policy_lookup
Tool Output:  Employees receive 20 paid leaves per year.
Final Answer: Employees are entitled to **20 paid leave days per year**.
Selected tool:  database_query
Tool Output:  Manish - Flutter Developer
Final Answer: Employee ID **101** belongs to **Manish**, who works as a **Flutter Developer**.
Selected tool:  company_policy_lookup
Tool Output:  Working hours are 9 AM to 6 PM.
Selected tool:  database_query
Tool Output:  Manish - Flutter Developer
Final Answer: **Working hours:** 9 AM – 6 PM.

**Employee ID 101:** Manish – Flutter Developer.


## Task 7: ReAct agent overview

ReAct Agent:
ReAct stands for Reasoning and Acting. It allows an AI agent to think about a task, use tools, and give an answer based on the results.

How does it work?

1. Reason: Understand the user's query.


2. Act: Select and use a tool.


3. Observe: Check the tool's output.


4. Repeat if needed.


5. Give the final answer.


A ReAct agent uses reasoning and tools to solve tasks step by step and thats why they are powerful.

## Task 8: Build a ReAct agent

In [ ]:
# As I have tried with ReAct style prompt I encoutered errors So I am following latest approach
# create_agent() API to build the AI agent.
from langchain.agents import create_agent
from langchain_core.tools import tool

@tool
def addition(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@tool
def multiplication(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

tools = [addition, multiplication, company_policy_lookup]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant. Use tools when needed."
)

response = agent.invoke({
    "messages": {"role": "user", "content": "What is 25 multiplied by 4?"}
})

print(response["messages"][-1].content)

25 multiplied by 4 equals **100**.


## Task 9: Test the ReAct agent

In [22]:
response = agent.invoke({
    "messages": {"role": "user", "content": "Add 25 and 15."}
})

print(response["messages"][-1].content)

response = agent.invoke({
    "messages": 
        {
            "role": "user",
            "content": "Multiply 10 by 5 and then add 20 to the result."
        }
})

print(response["messages"][-1].content)

The sum of 25 and 15 is **40**.
The result is **70**.


## Task 10: AI Agent Assistant mini-project

In [ ]:
# Please refer app.py

## Task 11: Observations and insights

1. Benefits of tool-augmented agents:
They can perform calculations, retrieve information, and automate tasks using tools.

2. Challenges with agents:
They may select incorrect tools, produce inaccurate answers, or face API errors.

3. Difference between chains and agents:
Chains follow a predefined sequence, while agents decide which tools to use based on the task.

4. When to use agents over RAG:
Use agents for tasks requiring tool selection and actions. Use RAG when answers need to be based on retrieved documents.